In [1]:
import warnings

warnings.filterwarnings("ignore")

import parmed as pmd
from pathlib import Path
import foyer
import json
import mbuild as mb
import gmso
from gmso.external import from_parmed

from parmed.exceptions import MoleculeError, FormatNotFound
from foyer.exceptions import MissingParametersError

import sTree

# Parsing OPLS

In [3]:
foyer_opls = foyer.forcefield.Forcefield(name="oplsaa")
opls_dir = Path("./files")

opls_atom_types = dict()
opls_bond_types = dict()
opls_angle_types = dict()
opls_harmonic_dih_types = dict()
opls_rb_dih_types = dict()
opls_improper_types = dict()
smart_strings = dict()
error_files = []
harm_and_rb_dihedral_files = []
harm_only_files = []

In [4]:
for top_file in opls_dir.glob("*.top"):
    found_harmonic_dih = False
    found_rb_dih = False
    try:
        struc = pmd.load_file(filename=str(top_file))
        # Parse atom types
        for atom in struc.atoms:
            opls_atom_types[atom.atom_type.name] = dict(
                sigma=atom.atom_type.sigma / 10,
                epsilon=atom.atom_type.epsilon,
                charge=atom.atom_type.charge,
            )
        
        # Parse harmonic bonds
        for bond in struc.bonds:
            bond_type = f"{bond.atom1.atom_type}-{bond.atom2.atom_type}"
            opls_bond_types[bond_type] = dict(req=bond.type.req / 10, k=bond.type.k)
        
        # Parse harmonic angles
        for angle in struc.angles:
            angle_type = f"{angle.atom1.atom_type}-{angle.atom2.atom_type}-{angle.atom3.atom_type}"
            opls_angle_types[angle_type] = dict(
                thetaeq=angle.type.theteq, k=angle.type.k
            )

        # Parse harmonic dihedrals
        for dih in struc.dihedrals:
            dihedral_type = f"{dih.atom1.atom_type}-{dih.atom2.atom_type}-{dih.atom3.atom_type}-{dih.atom4.atom_type}"
            opls_harmonic_dih_types[dihedral_type] = dict(
                k=dih.type.phi_k, phase=dih.type.phase
            )
            found_harmonic_dih = True

        # Parse RB dihedrals
        for rb in struc.rb_torsions:
            rb_type = f"{rb.atom1.atom_type}-{rb.atom2.atom_type}-{rb.atom3.atom_type}-{rb.atom4.atom_type}"
            opls_rb_dih_types[rb_type] = dict(
                c0=rb.type.c0,
                c1=rb.type.c1,
                c2=rb.type.c2,
                c3=rb.type.c3,
                c4=rb.type.c4,
                c5=rb.type.c5,
            )
            found_rb_dih = True
    except (MoleculeError, ValueError, FormatNotFound) as e:
        print(f"Error: {e}")
        error_files.append(str(top_file))

    if all([found_harmonic_dih, found_rb_dih]):
        harm_and_rb_dihedral_files.append(str(top_file))
    elif found_harmonic_dih is True and found_rb_dih is False:
        harm_only_files.append(str(top_file))

Error: Cannot exclude an atom from itself! Atoms are: <Atom N [5]; In MOL 5> <Atom N [5]; In MOL 5>
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [12]; In MOL 0> <Atom N [12]; In MOL 0>
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [6]; In MOL 6> <Atom N [6]; In MOL 6>
Error: invalid literal for int() with base 10: 'formamide'
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [8]; In MOL 0> <Atom N [8]; In MOL 0>
Error: Cannot exclude an atom from itself! Atoms are: <Atom N [14]; In MOL 14> <Atom N [14]; In MOL 14>


In [6]:
# Look through ffnonbonded.itp to get classes for the new atom types in the first dict
# Try reverse for angles, bonds and dihedrals

new_atom_types = dict()
new_bond_types = dict()
new_angle_types = dict()
new_harmonic_dih_types = dict()
new_rb_dih_types = dict()

for atom_type in opls_atom_types:
    if atom_type not in foyer_opls.atomTypeDefinitions.keys():
        new_atom_types[atom_type] = opls_atom_types[atom_type]

for bond_type in opls_bond_types:
    atom1 = bond_type.split("-")[0]
    atom2 = bond_type.split("-")[1]
    class1 = foyer_opls.atomTypeClasses[atom1]
    class2 = foyer_opls.atomTypeClasses[atom2]
    found = False
    for _bond in [[atom1, atom2], [atom2, atom1], [class1, class2], [class2, class1]]:
        try:
            bond_params = foyer_opls.get_parameters(
                group="harmonic_bonds", key=[_bond[0], _bond[1]]
            )
            found = True
        except MissingParametersError:
            pass
    if not found:
        new_bond_types[bond_type] = opls_bond_types[bond_type]

for angle_type in opls_angle_types:
    atom1 = angle_type.split("-")[0]
    atom2 = angle_type.split("-")[1]
    atom3 = angle_type.split("-")[2]
    class1 = foyer_opls.atomTypeClasses[atom1]
    class2 = foyer_opls.atomTypeClasses[atom2]
    class3 = foyer_opls.atomTypeClasses[atom3]
    found = False
    for _angle in [[atom1, atom2, atom3], [atom3, atom2, atom1], [class1, class2, class3], [class3, class2, class1]]:
        try:
            angle_params = foyer_opls.get_parameters(
                group="harmonic_angles", key=[_angle[0], _angle[1], _angle[2]]
            )
            found = True
        except MissingParametersError:
            pass
    if not found:
        new_angle_types[angle_type] = opls_angle_types[angle_type]

for rb_type in opls_rb_dih_types:
    atom1 = rb_type.split("-")[0]
    atom2 = rb_type.split("-")[1]
    atom3 = rb_type.split("-")[2]
    atom4 = rb_type.split("-")[3]
    class1 = foyer_opls.atomTypeClasses[atom1]
    class2 = foyer_opls.atomTypeClasses[atom2]
    class3 = foyer_opls.atomTypeClasses[atom3]
    class4 = foyer_opls.atomTypeClasses[atom4]
    found = False
    for _rb in [
        [atom1, atom2, atom3, atom4],
        [atom4, atom3, atom2, atom1],
        [class1, class2, class3, class4],
        [class4, class3, class2, class1]
    ]:
        try:
            params = foyer_opls.get_parameters(
                group="rb_propers", key=[_rb[0], _rb[1], _rb[2], _rb[3]]
            )
            found = True
        except MissingParametersError:
            pass
    if not found:
        new_rb_dih_types[rb_type] = opls_rb_dih_types[rb_type]

In [7]:
print(f"New Atom Types: {len(new_atom_types)}")
print(f"New Bond Types: {len(new_bond_types)}")
print(f"New Angle Types: {len(new_angle_types)}")
print(f"New RB Tors Types: {len(new_rb_dih_types)}")

New Atom Types: 73
New Bond Types: 153
New Angle Types: 329
New RB Tors Types: 444


# SMARTS DEFS

In [ ]:
files_dict = {}

for top_file in opls_dir.glob("*.top"):
    if str(top_file) in error_files:
        continue
    base_name = top_file.stem
    files_dict[str(top_file)] = []
    # Find matching pdb files
    matching_pdbs = list(opls_dir.glob(f"{base_name}-*.pdb"))
    for pdb_file in matching_pdbs:
        files_dict[str(top_file)].append(str(pdb_file))

In [ ]:
compounds = []
top_files = []
pdb_files = []

for top_file in files_dict:
    if top_file in error_files:
        continue
    try:
        pdb_file = files_dict[top_file][0]
        comp = mb.load(pdb_file)
        child = comp.children[0]
        compounds.append(child)
        top_files.append(top_file)
        pdb_files.append(pdb_file)
    except:
        print(top_file)

In [ ]:
for comp, top_file in zip(compounds, top_files):
    struc = comp.to_parmed()
    top = pmd.load_file(top_file)
    top.coordinates = struc.coordinates
    gmso_top = from_parmed(top)
    break
    

In [19]:
comp = mb.load("CCC", smiles=True)

In [20]:
for p in comp.particles_by_name("C"):
    p.name = "_X"
    p.element = None

hs = [p for p in comp.particles_by_name("H")]
for h in hs:
    comp.remove(h)

In [21]:
BG = comp.bond_graph # the bond graph of our molecule, atoms connected by bonds represented as a Set of source atom to destination atoms

depth = 1 # this parameter determines how large the smart tree should be generated, the larger the depth the more specific your SMARTS definition is, but the more expensive it is to atomtype 
smarts_dict = sTree.bond_graph_to_smarts_dic(BG, depth) # returns our smarts in a dictionary

AttributeError: 'NoneType' object has no attribute 'symbol'

In [17]:
for i in smarts_dict.values():
    if i in foyer_opls.atomTypeDefinitions:
        print(i, "Already in Foyer")
    else:
        print(i, "Not in foyer")

[C;X1](C) Not in foyer
[C;X2](C)(C) Not in foyer
[C;X1](C) Not in foyer
